# Shadow training — Colab Pro launcher

Mount Drive (so checkpoints persist), clone the repo + dependencies, generate the Shadow datasets, and kick off training. Run cells top-to-bottom.

Recommended runtime: Colab Pro **L4 GPU** or **A100**. 9 training runs (3 variants × 3 seeds) at 5000 epochs each take ≈ 60–90 GPU-hours total.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
!mkdir -p shadow-project && cd shadow-project

In [ ]:
%cd /content/drive/MyDrive/shadow-project
![ -d robosuite ]    || git clone https://github.com/ARISE-Initiative/robosuite.git    && cd robosuite && git checkout v1.5.1 && cd ..
![ -d robomimic ]    || git clone https://github.com/ARISE-Initiative/robomimic.git
![ -d mimicgen ]     || git clone https://github.com/NVlabs/mimicgen.git
![ -d shadow ]       || git clone <YOUR_REPO_URL> shadow

In [ ]:
!pip install -q mujoco==3.2.7 cmake
!cd robosuite && pip install -e . -q
!cd robomimic && pip install -e . -q     # remember the egl_probe + env_robosuite patches if re-cloning
!cd mimicgen && pip install -e . -q
!pip install -q seaborn scipy
!cd robosuite && python robosuite/scripts/setup_macros.py

In [ ]:
%cd /content/drive/MyDrive/shadow-project/shadow
# add the compat shim and the egl_probe/env_robosuite patches if they aren't already in place — see CLAUDE.md
!python mimicgen/mimicgen/scripts/download_datasets.py --download_dir datasets --dataset_type core --tasks stack_d0

In [ ]:
# Build the Shadow datasets (~1 hour each on Colab — pass 1 + pass 2 + write).
!python create_shadow_dataset.py --src datasets/core/stack_d0.hdf5 --out datasets/shadow/stack_d0_shadow.hdf5       --source_robot Panda --target_robot Sawyer --noise none
!python create_shadow_dataset.py --src datasets/core/stack_d0.hdf5 --out datasets/shadow/stack_d0_shadow_noise.hdf5 --source_robot Panda --target_robot Sawyer --noise gaussian --sigma_xyz 0.01 --sigma_rot_deg 5.0

In [ ]:
# Train! Each run is ~6–10h on L4. Launch one variant×seed at a time so you can resume on Colab disconnects.
import subprocess
VARIANTS = ['rgb', 'shadow_vanilla', 'shadow_noise']
SEEDS = [0, 1, 2]
for v in VARIANTS:
    for s in SEEDS:
        print(f'=== training {v} seed {s} ===')
        subprocess.run(['python', 'train.py', '--variant', v, '--seed', str(s)], check=True)